In [40]:
# clustering_module.py

import pandas as pd
import numpy as np

from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

import plotly.express as px

def run_clustering(
    attendance_path: str,
    grades_path: str,
    cluster_method: str = "kmeans",
    top_grade_threshold: float = 4.5,
    top_attendance_threshold: float = 90,
    good_attendance_threshold: float = 85,
    low_attendance_threshold: float = 50,
    show_figures: bool = True
) -> dict:
    """
    Запуск анализа и кластеризации студентов по среднему баллу и посещаемости.

    Параметры:
        attendance_path (str): путь к CSV с посещаемостью
        grades_path (str): путь к CSV с результатами студентов
        cluster_method (str): метод кластеризации ("kmeans", "dbscan", "agglomerative")
        top_grade_threshold (float): порог для отличников
        top_attendance_threshold (float): порог для активных студентов
        good_attendance_threshold (float): порог для хорошей посещаемости
        low_attendance_threshold (float): порог для низкой посещаемости
        show_figures (bool): строить интерактивные графики

    Возвращает:
        dict с ключами:
            "data"              : DataFrame с результатами и кластерами
            "group_stats"       : средние показатели по группам
            "group_activity"    : активность групп (кол-во студентов и средняя посещаемость)
            "top_students"      : топ-10 активных отличников
            "good_attendance"   : студенты с хорошей посещаемостью
            "low_attendance"    : студенты с низкой посещаемостью
    """

    # --- 1. Загрузка данных ---
    attendance = pd.read_csv(attendance_path)
    grades = pd.read_csv(grades_path)

    # --- 2. Фильтр по институту ---
    grades = grades[
        (grades['Faculty'] == 'Институт информационных технологий и анализа данных') |
        (grades['Faculty_ID'] == 46.0)
    ]

    # --- 3. Подготовка данных ---
    attendance = attendance[['mira_id','lesson_id','value']]
    grades = grades[['Student_ID','Group','Result']]

    # --- 4. Посещаемость ---
    attendance["attended_flag"] = attendance["value"].notna().astype(int)
    attendance_stats = attendance.groupby('mira_id').agg(
        total_lessons=('lesson_id','count'),
        attended=('attended_flag','sum')
    )
    attendance_stats["attendance_percent"] = (
        attendance_stats["attended"] / attendance_stats["total_lessons"] * 100
    )
    attendance_stats = attendance_stats.reset_index()

    # --- 5. Средний балл ---
    grades["Result"] = pd.to_numeric(grades["Result"], errors="coerce")
    grades_stats = grades.groupby(
        ["Student_ID","Group"]
    ).agg(avg_grade=("Result","mean")).reset_index()

    # --- 6. Объединение ---
    data = pd.merge(
        grades_stats,
        attendance_stats,
        left_on="Student_ID",
        right_on="mira_id"
    )
    data = data[["Student_ID","Group","avg_grade","attendance_percent"]].dropna()

    # --- 7. Масштабирование ---
    features = data[["avg_grade","attendance_percent"]]
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(features)

    # --- 8. Кластеризация ---
    if cluster_method.lower() == "kmeans":
        silhouette_scores = []
        K = range(2,10)
        for k in K:
            kmeans = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=42)
            labels = kmeans.fit_predict(X_scaled)
            score = silhouette_score(X_scaled, labels)
            silhouette_scores.append(score)
        best_k = K[np.argmax(silhouette_scores)]
        model = KMeans(n_clusters=best_k, init="k-means++", random_state=42)
        data["cluster"] = model.fit_predict(X_scaled)
    elif cluster_method.lower() == "dbscan":
        model = DBSCAN(eps=0.5, min_samples=10)
        data["cluster"] = model.fit_predict(X_scaled)
    elif cluster_method.lower() == "agglomerative":
        model = AgglomerativeClustering(n_clusters=3)
        data["cluster"] = model.fit_predict(X_scaled)
    else:
        raise ValueError("Неверный метод кластеризации")

    # Преобразуем кластер в категорию для графиков
    data["cluster_str"] = data["cluster"].astype(str)

    # --- 9. Интерактивный график института ---
    # --- 9. Интерактивный график института ---
    if show_figures:
        cluster_colors = px.colors.qualitative.Dark2
        # Создаём соответствие цвета для всех кластеров
        unique_clusters = sorted(data["cluster"].unique())
        color_map = {str(cluster): cluster_colors[i % len(cluster_colors)]
                    for i, cluster in enumerate(unique_clusters)}

        fig = px.scatter(
            data,
            x="attendance_percent",
            y="avg_grade",
            color="cluster_str",
            color_discrete_map=color_map,  # используем маппинг
            hover_data=["Student_ID","Group"],
            title="Кластеры студентов ИТИАД по группам"
        )
        fig.show()

    # --- 10. Анализ по группам ---
    group_stats = data.groupby("Group")[["avg_grade","attendance_percent"]].mean()

    group_activity = data.groupby("Group").agg(
        students_count=("Student_ID", "nunique"),
        avg_attendance=("attendance_percent", "mean")
    ).sort_values(by=["students_count", "avg_attendance"], ascending=False)

    # --- 11. Топ-10 самых активных отличников ---
    top_students = data[
        (data["avg_grade"] >= top_grade_threshold) &
        (data["attendance_percent"] >= top_attendance_threshold)
    ].sort_values(by=["avg_grade","attendance_percent"], ascending=False).head(10)

    # --- 12. Студенты с хорошей посещаемостью ---
    good_attendance = data[
        (data["attendance_percent"] >= good_attendance_threshold)
    ].sort_values(by="attendance_percent", ascending=False)

    # --- 13. Студенты с низкой посещаемостью ---
    low_attendance = data[
        (data["attendance_percent"] < low_attendance_threshold)
    ].sort_values(by="attendance_percent")

     # --- 14. Графики по группам ---
    # --- 14. Графики по группам ---
    if show_figures:
        groups = data["Group"].unique()
        for group in groups:
            subset = data[data["Group"] == group]
            fig_group = px.scatter(
                subset,
                x="attendance_percent",
                y="avg_grade",
                color="cluster_str",
                color_discrete_map=color_map,  # та же карта цветов
                hover_data=["Student_ID"],
                title=f"Кластеры группы {group} (всего {subset['Student_ID'].nunique()} студентов)"
            )
            fig_group.show()


    return {
        "data": data,
        "group_stats": group_stats,
        "group_activity": group_activity,
        "top_students": top_students,
        "good_attendance": good_attendance,
        "low_attendance": low_attendance
    }


results = run_clustering(
    attendance_path="/content/drive/MyDrive/проект обработка данных/attendance_all.csv",
    grades_path="/content/drive/MyDrive/проект обработка данных/export_studs_cleaned.csv",
    cluster_method="kmeans",
    top_grade_threshold=4.5,
    top_attendance_threshold=90,
    good_attendance_threshold=75,
    low_attendance_threshold=50,
    show_figures=True
)

# Доступ к результатам
data = results["data"]
group_stats = results["group_stats"]
group_activity = results["group_activity"]
top_students = results["top_students"]
good_attendance = results["good_attendance"]
low_attendance = results["low_attendance"]

# Просмотр первых строк таблицы
print("=== Объединённые данные с кластерами ===")
print(data.head())
print("\n=== Средние показатели по группам ===")
print(group_stats.head())
print("\n=== Активность групп ===")
print(group_activity.head())
print("\n=== Топ-10 активных отличников ===")
print(top_students)
print("\n=== Студенты с хорошей посещаемостью ===")
print(good_attendance.head())
print("\n=== Студенты с низкой посещаемостью ===")
print(low_attendance.head())


=== Объединённые данные с кластерами ===
   Student_ID    Group  avg_grade  attendance_percent  cluster cluster_str
0     2373007  ИСТб-22   4.250000           100.00000        0           0
1     2391969  ИСТб-22   4.454545            89.13738        2           2
2     2409774  ЭВМб-24   3.454545           100.00000        0           0
3     2410815   ИБб-22   3.466667            50.00000        1           1
4     2418744  АСУб-21   3.583333           100.00000        0           0

=== Средние показатели по группам ===
         avg_grade  attendance_percent
Group                                 
АСУб-21   3.952442           80.676237
АСУб-22   4.477533           86.669029
АСУб-23   4.472758           85.957165
АСУб-24   4.078205           84.681779
БКСм-23   3.555556          100.000000

=== Активность групп ===
         students_count  avg_attendance
Group                                  
ИСТб-24              60       85.447429
ИСТб-22              47       89.146496
ИСТб-23    